# Save and load the model

A trained model is worth nothing if it disappears when the kernel restarts. Two things can be saved:

- the **weights**, which is the normal way
- the **whole object**, which is convenient and fragile

Files go into `models/`, which is ignored by git.

In [ ]:
from pathlib import Path

import torch
from torch import nn

torch.manual_seed(42)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.linear_relu_stack(self.flatten(x))


model = NeuralNetwork().to(device)

## state_dict

A plain dictionary mapping each parameter name to its tensor. It holds the learned values and nothing else.

The architecture is not in there. That lives in the Python class, which is why the class is still needed to load weights back.

In [ ]:
state = model.state_dict()

print(type(state).__name__)

for key, value in state.items():
    print(key, tuple(value.shape), value.dtype)

## Saving the weights

The recommended way. The file holds tensors only, so it stays valid even if the class is later moved or renamed.

`.pth` and `.pt` are both just conventions.

In [ ]:
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
weights_path = model_dir / "mlp_weights.pth"

torch.save(model.state_dict(), weights_path)

print(weights_path.exists())
print(round(weights_path.stat().st_size / 1024**2, 2), "MB")

## Loading the weights

Two steps, and the order matters:

1. build an empty model of the same class, which creates the layers with random values
2. overwrite them with `load_state_dict`

`weights_only=True` allows only tensors to be unpickled. It blocks arbitrary code hidden in a checkpoint file, and it is the default in current PyTorch. Keep it on for anything downloaded.

`map_location` decides where the tensors land. Without it, a file saved on CUDA fails to load on a machine that has no GPU.

In [ ]:
loaded = NeuralNetwork()
result = loaded.load_state_dict(
    torch.load(weights_path, map_location=device, weights_only=True)
)
loaded = loaded.to(device)

print(result)

The real check is that both models produce the same numbers for the same input.

In [ ]:
x = torch.rand(4, 28, 28, device=device)

model.eval()
loaded.eval()

with torch.no_grad():
    original_out = model(x)
    loaded_out = loaded(x)

print(torch.allclose(original_out, loaded_out))
print((original_out - loaded_out).abs().max().item())

## eval before inference

`model.eval()` must be called before predicting. A freshly loaded model is in train mode, so dropout would still drop values and batch norm would still use batch statistics. The predictions would be random and quietly wrong.

`torch.no_grad()` is the other half: no graph, less memory, faster.

In [ ]:
loaded.eval()

with torch.no_grad():
    logits = loaded(x)
    probabilities = logits.softmax(dim=1)
    predicted = probabilities.argmax(dim=1)

print(logits.shape)
print(predicted)
print(loaded.training)

## Saving the whole object

This looks easier, because no class is needed at load time. It is not.

Pickle stores a **reference** to the class, not its source. Loading needs that exact class importable from the exact same module path. Rename the file, move the class, or reorganise the package and the checkpoint stops loading.

It also needs `weights_only=False`, which will execute whatever is inside the file. Never do that with a file from someone else.

In [ ]:
full_path = model_dir / "mlp_full.pth"
torch.save(model, full_path)

restored = torch.load(full_path, map_location=device, weights_only=False)
restored.eval()

with torch.no_grad():
    restored_out = restored(x)

print(torch.allclose(original_out, restored_out))

## Checkpoints for resuming training

Weights alone are enough for inference, but not to continue training. The optimizer carries state of its own: momentum buffers for SGD, the moment estimates for Adam. Dropping it restarts the optimizer cold and the loss jumps.

A checkpoint is an ordinary dict, so anything can go in it.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
checkpoint_path = model_dir / "checkpoint.pth"

torch.save(
    {
        "epoch": 5,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "test_loss": 1.0905,
    },
    checkpoint_path,
)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

print(sorted(checkpoint.keys()))
print(checkpoint["epoch"], checkpoint["test_loss"])

In [ ]:
resumed = NeuralNetwork().to(device)
resumed_optimizer = torch.optim.SGD(resumed.parameters(), lr=1e-3, momentum=0.9)

resumed.load_state_dict(checkpoint["model_state"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

resumed.eval()
with torch.no_grad():
    resumed_out = resumed(x)

print(torch.allclose(original_out, resumed_out))
print(f"resume from epoch {start_epoch}")

## Summary

```text
inference only:      state_dict
resume training:     state_dict plus optimizer state plus epoch
share with someone:  state_dict, never the pickled object
```

Three habits worth keeping: save the `state_dict`, load with `weights_only=True`, and call `eval()` before predicting.